# Single Frame Extractor

Extract **one frame** from a video at a specific timestamp.

**Input:** original video or a clip from Scene_Clip_Extraction  
**Output:** single JPEG → Google Drive

**Filename format** (same as Frame_Extractor):
`{video_stem}_{frame_index:07d}_t{time_ms:08d}ms.jpg`

Example: `M01_white_1080p_clip_001_00-14_0000135_t00004500ms.jpg`  
→ **4500 ms** (4.5 s), frame index **135**

Set `TARGET_TIME_MS = 4500` or `'t00004500ms'`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
import re
from pathlib import Path

# ── Input: single video or folder of clips ────────────────────────────────────
# Choose one:
INPUT_VIDEO  = '/content/drive/MyDrive/bradford_bulls/clips/M01_white_1080p/M01_white_1080p_clip_001_00-14.mp4'
INPUT_FOLDER = ''   # leave empty to use INPUT_VIDEO
                    # or: '/content/drive/MyDrive/bradford_bulls/clips/M01_white_1080p'

# ── Timestamp ───────────────────────────────────────────────────────────────
# Option A: milliseconds — same format as output filename suffix
#   int:  4500
#   str:  '4500' | '4500ms' | 't00004500ms'
TARGET_TIME_MS = 4500

# Option B: parse from an existing frame filename (overrides TARGET_TIME_MS)
# Example: 'M01_white_1080p_clip_001_00-14_0000135_t00004500ms.jpg'
REFERENCE_FRAME = ''

# ── Output ──────────────────────────────────────────────────────────────────
OUTPUT_ROOT  = Path('/content/drive/MyDrive/bradford_bulls/frames')
JPEG_QUALITY = 95

# ── Resolve input files + source name ────────────────────────────────────────
# Output structure (same as Frame_Extractor):
#   INPUT_VIDEO  → OUTPUT_ROOT / video_stem / frame.jpg
#   INPUT_FOLDER → OUTPUT_ROOT / folder_stem / clip_stem / frame.jpg
if INPUT_FOLDER:
    video_files = sorted(Path(INPUT_FOLDER).glob('*.mp4'))
    source_name = Path(INPUT_FOLDER).stem
    use_subdir  = True
else:
    video_files = [Path(INPUT_VIDEO)]
    source_name = Path(INPUT_VIDEO).stem
    use_subdir  = False

def parse_time_ms(value) -> int:
    """Accept 4500, '4500', '4500ms', 't00004500ms', or full frame filename."""
    if isinstance(value, str):
        m = re.search(r'_t(\d+)ms', value) or re.search(r't(\d+)ms', value, re.I) or re.search(r'^(\d+)\s*ms?$', value.strip(), re.I)
        if m:
            return int(m.group(1))
        return int(value.strip())
    return int(value)

if REFERENCE_FRAME:
    target_time_ms = parse_time_ms(Path(REFERENCE_FRAME).name)
    time_source = f'REFERENCE_FRAME → {Path(REFERENCE_FRAME).name}'
else:
    target_time_ms = parse_time_ms(TARGET_TIME_MS)
    time_source = f'TARGET_TIME_MS = {target_time_ms}  (t{target_time_ms:08d}ms)'

target_time_s = target_time_ms / 1000.0

print(f'Input videos  : {len(video_files)}')
for v in video_files:
    print(f'  {v.name}')
print(f'\nSource name   : {source_name}')
print(f'Timestamp     : t{target_time_ms:08d}ms  ({target_time_s:.3f} s)')
print(f'  ({time_source})')
print(f'Output root   : {OUTPUT_ROOT / source_name}')

## 3. Preview — frame index & expected filename

In [ ]:
import cv2

print(f'{"File":<45} {"Duration":>10} {"FPS":>6} {"Frame idx":>10} {"Expected filename"}')
print('-' * 110)

preview_rows = []

for vf in video_files:
    cap = cv2.VideoCapture(str(vf))
    vid_fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = n_frames / vid_fps
    cap.release()

    frame_idx = min(int(round(target_time_ms / 1000.0 * vid_fps)), max(0, n_frames - 1))
    t_ms      = int(frame_idx / vid_fps * 1000)
    fname     = f'{vf.stem}_{frame_idx:07d}_t{t_ms:08d}ms.jpg'

    if target_time_ms / 1000.0 > duration:
        warn = f' ⚠ t > duration ({duration:.2f}s) — will use last frame'
    else:
        warn = ''

    preview_rows.append((vf, frame_idx, fname, vid_fps, duration, warn))
    dur_str = f'{duration:.2f}s'
    print(f'{vf.name:<45} {dur_str:>10} {vid_fps:>6.1f} {frame_idx:>10} {fname}{warn}')

print('-' * 110)

## 4. Extract frame at t

In [ ]:
import cv2
from pathlib import Path

saved = []

for vf, frame_idx, expected_fname, vid_fps, duration, _ in preview_rows:
    cap = cv2.VideoCapture(str(vf))
    vid_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_idx = min(int(round(target_time_ms / 1000.0 * vid_fps)), max(0, n_total - 1))

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()

    if not ok:
        print(f'  ✗ Failed to read frame from {vf.name}')
        continue

    t_ms = int(frame_idx / vid_fps * 1000)
    fname = f'{vf.stem}_{frame_idx:07d}_t{t_ms:08d}ms.jpg'

    if use_subdir:
        out_dir = OUTPUT_ROOT / source_name / vf.stem
    else:
        out_dir = OUTPUT_ROOT / source_name
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / fname
    cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    saved.append(out_path)
    print(f'  ✓ {vf.name} → {out_path}')

print(f'\nSaved {len(saved)} frame(s)')

## 5. View extracted frame

In [ ]:
import matplotlib.pyplot as plt
import cv2

if not saved:
    print('No frames saved.')
else:
    n = len(saved)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), squeeze=False)
    axes = axes.flatten()

    for i, fp in enumerate(saved):
        img = cv2.imread(str(fp))
        m = re.search(r'_(\d{7})_t(\d+)ms', fp.name)
        idx_str = m.group(1) if m else '?'
        ms_str  = m.group(2) if m else '?'
        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f'{fp.name}\nframe {idx_str} @ {ms_str} ms', fontsize=9)
        axes[i].axis('off')

    plt.suptitle(f'Frame at t{target_time_ms:08d}ms ({target_time_s:.3f} s)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()